<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/10_linguistic_analysis_with_spacy_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Linguistic Analysis with spaCy · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. Your numbers will differ a little from run to run, so read the comments as "what
to look for", not as exact results.

Most of this course treats text as a bag of words or as vectors. But language has structure: words
have grammatical roles, sentences have subjects and objects, and some words name people, places and
organizations. **spaCy** is the standard Python library for analyzing that structure quickly, in
many languages.

This matters for social science because structure answers questions bags of words can't: not just
*whether* a speech mentions the government, but whether the government is the one *acting* or being
*acted upon*; not just how often "we" appears, but what "we" *will do*.

In this notebook you will:

- run spaCy's pipeline and look inside the `Doc`, `Token` and `Span` objects
- compare tokenization, lemmatization and stemming
- use part-of-speech tags and named entities to describe 60 US inaugural addresses (1789-2025)
- extract "who does what to whom" from dependency parses
- find patterns such as "we will + verb" with the rule-based `Matcher`
- analyze Dutch with the same code
- process large corpora efficiently

**Contents**

1. The pipeline
2. Tokens and their attributes
3. Lemmas versus stems
4. A corpus: 60 inaugural addresses
5. Parts of speech as measurements
6. Named entities
7. Dependencies: who does what to whom
8. Rule-based matching
9. Dutch, and working at scale

No GPU needed.

## 0. Setup

spaCy comes with Colab; the language models are downloaded separately. `sm` models are small and
fast; `md`/`lg` add word vectors and are a bit more accurate; `trf` models use a transformer (like
Day 3's BERT) and are the most accurate, but slow without a GPU.

In [ ]:
!pip install -q spacy nltk
!python -m spacy download en_core_web_sm -q
!python -m spacy download nl_core_news_sm -q

import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import spacy
from nltk.stem import PorterStemmer
from spacy import displacy
from spacy.matcher import Matcher, PhraseMatcher

nlp = spacy.load("en_core_web_sm")
pd.set_option("display.max_colwidth", 100)
print("spaCy", spacy.__version__, "| pipeline:", nlp.pipe_names)

## 1. The pipeline

Calling `nlp(text)` runs the text through a **pipeline** of components, each adding information:

| Component | Adds |
|---|---|
| tokenizer | splits text into tokens (always first) |
| `tok2vec` | a vector per token that the next components share |
| `tagger` | detailed part-of-speech tags (`NN`, `VBD`, ...) |
| `parser` | the dependency tree, and sentence boundaries |
| `attribute_ruler`, `lemmatizer` | coarse part of speech (`NOUN`, `VERB`, ...) and lemmas |
| `ner` | named entities |

The result is a **`Doc`**: a sequence of **`Token`**s, with **`Span`**s (sentences, entities, noun
phrases) on top.

In [ ]:
doc = nlp("The new government didn't explain the nitrogen rules to farmers in Gelderland, "
          "and on Tuesday thousands of them drove their tractors to The Hague.")

print("Tokens:   ", [t.text for t in doc])
print("Sentences:", len(list(doc.sents)))
print("Entities: ", [(e.text, e.label_) for e in doc.ents])

Note how the tokenizer splits "didn't" into "did" and "n't", and keeps "The Hague" as two tokens but
one entity. Compare with plain `str.split()`, which would give you `"didn't"` and `"Gelderland,"` (with
the comma) as words.

Also note that the small English model may well tag *Gelderland* as a person: it has seen few Dutch
province names. NER is a model, and models make mistakes, especially on names that were rare in their
training data.

## 2. Tokens and their attributes

Every token carries a lot of information. Attributes ending in `_` are readable strings (`pos_`),
the same names without `_` are internal numbers.

In [ ]:
pd.DataFrame([{
    "text": t.text, "lemma": t.lemma_, "pos": t.pos_, "tag": t.tag_, "dep": t.dep_, "head": t.head.text,
    "stop": t.is_stop, "punct": t.is_punct, "shape": t.shape_, "entity": t.ent_type_,
} for t in doc])

- `pos_`: the coarse, universal part of speech (`NOUN`, `VERB`, `ADJ`, `PRON`, `PROPN` for proper
  nouns, ...). `spacy.explain("PROPN")` explains any label.
- `tag_`: the detailed, language-specific tag (`VBD` = verb, past tense).
- `dep_` and `head`: the token's grammatical relation to another token (Section 7).
- `is_stop`, `is_punct`, `like_num`: handy filters for preprocessing.
- `shape_`: the token's "look" (`Xxxxx` for a capitalized word, `dd` for two digits).

> **✏️ Exercise 1**
>
> Pick a sentence from your own research area (or a news headline) and:
>
> 1. Show its tokens with lemma, part of speech and dependency in a DataFrame.
> 2. List the tokens whose lemma differs from their lowercased text. What kinds of words are they?
> 3. Use `spacy.explain` to look up two labels you don't know.

In [ ]:
# ✅ Solution
my_doc = nlp("Researchers found that younger voters were sharing more political memes than older voters did.")
display(pd.DataFrame([{"text": t.text, "lemma": t.lemma_, "pos": t.pos_, "dep": t.dep_} for t in my_doc]))
print("Lemma differs:", [(t.text, t.lemma_) for t in my_doc if t.lemma_ != t.text.lower()])
print("acomp:", spacy.explain("acomp"), "| ccomp:", spacy.explain("ccomp"), "| JJR:", spacy.explain("JJR"))
# Inflected forms: plurals (voters -> voter, memes -> meme), past tenses (found -> find, were -> be,
# did -> do), comparatives (younger -> young), -ing forms (sharing -> share).

## 3. Lemmas versus stems

Day 1 reduced words to their **stem** with a rule-based stemmer: it chops off endings, without knowing
the word. A **lemma** is the dictionary form, found using the word's part of speech and a lookup
table. Lemmas are real words; stems often aren't, and stemmers lump together words with different
meanings.

In [ ]:
stemmer = PorterStemmer()
words = "studies studying policies universities better was organization organized general generous news".split()
lemmas = {t.text: t.lemma_ for t in nlp(" ".join(words))}
pd.DataFrame({"word": words, "Porter stem": [stemmer.stem(w) for w in words],
              "spaCy lemma": [lemmas[w] for w in words]})

Lemmas depend on context: "saw" is *see* as a verb but *saw* as a noun, and spaCy decides from the
sentence. (Out of context, as in the list above, it has to guess the part of speech.) Stemming is
faster and fine for many bag-of-words tasks; lemmatizing is better when you'll read the words, and
for languages with rich inflection, like Dutch.

## 4. A corpus: 60 inaugural addresses

NLTK includes every US presidential inaugural address from Washington (1789) to Trump (2025). For a
corpus, `nlp.pipe` is much faster than calling `nlp` on each text, because it processes texts in
batches.

In [ ]:
nltk.download("inaugural", quiet=True)
from nltk.corpus import inaugural

speeches = pd.DataFrame([{"year": int(f[:4]), "president": f[5:-4], "text": inaugural.raw(f)}
                         for f in inaugural.fileids()])
print(len(speeches), "speeches,", speeches.text.str.split().str.len().sum(), "words")

start = time.time()
docs = list(nlp.pipe(speeches["text"], batch_size=8))
print(f"Processed in {time.time() - start:.0f} s")
speeches.tail(3)

## 5. Parts of speech as measurements

Part-of-speech counts are simple stylistic measures. Here: the share of each part of speech per speech,
and how the *length of sentences* has changed. Long, complex sentences were a sign of a formal style;
modern speeches are written to be heard.

In [ ]:
rows = []
for year, doc in zip(speeches["year"], docs):
    words = [t for t in doc if not t.is_punct and not t.is_space]
    pos = Counter(t.pos_ for t in words)
    sentences = list(doc.sents)
    rows.append({"year": year, "words": len(words),
                 "words per sentence": len(words) / len(sentences),
                 **{p: pos[p] / len(words) for p in ["NOUN", "VERB", "ADJ", "PRON", "PROPN"]}})
style = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(style.year, style["words per sentence"], "o-", color="#34B233", markersize=3)
axes[0].set_title("Average sentence length (words)")
for p, c in [("NOUN", "#1A1A2E"), ("PRON", "#34B233"), ("ADJ", "#6B7280")]:
    axes[1].plot(style.year, style[p].rolling(5, center=True).mean(), label=p, color=c)
axes[1].set_title("Share of words (5-speech rolling average)"); axes[1].legend()
plt.tight_layout(); plt.show()

Sentences have become much shorter over two centuries: from more than 40 words on average before
1850 to fewer than 20 since 2000. Measures like these are crude, and spaCy's
parser was trained on modern English, so its sentence boundaries and tags are less reliable for 18th
century prose. But the overall trend is large enough to survive that.

> **✏️ Exercise 2**
>
> Presidents speak for the nation as "we", or for themselves as "I". For each speech, count the
> first-person plural pronouns (`we`, `us`, `our`, `ours`, `ourselves`) and singular ones (`i`, `me`,
> `my`, `mine`, `myself`) per 1,000 words, using `token.lower_` and `token.pos_ == "PRON"`. Plot both over
> time. Which five speeches use "we" the most?

In [ ]:
# ✅ Solution
WE = {"we", "us", "our", "ours", "ourselves"}
I = {"i", "me", "my", "mine", "myself"}
rows = []
for (year, president), doc in zip(speeches[["year", "president"]].values, docs):
    words = [t for t in doc if not t.is_punct and not t.is_space]
    prons = [t.lower_ for t in words if t.pos_ == "PRON"]
    rows.append({"year": year, "president": president,
                 "we": 1000 * sum(p in WE for p in prons) / len(words),
                 "I": 1000 * sum(p in I for p in prons) / len(words)})
persons = pd.DataFrame(rows)

plt.figure(figsize=(10, 3.5))
plt.plot(persons.year, persons["we"], "o-", label='"we"', color="#34B233", markersize=3)
plt.plot(persons.year, persons["I"], "o-", label='"I"', color="#1A1A2E", markersize=3)
plt.ylabel("per 1,000 words"); plt.legend(); plt.show()
display(persons.nlargest(5, "we").round(1))
# "We" rises strongly in the 20th century (from under 20 per 1,000 words before 1900 to about 60
# since 1950) while "I" roughly halves: the president speaking for the collective rather than about
# his own role. (spaCy tags possessives like "my" and "our" as PRON; check a few examples like this
# before trusting a count.)

## 6. Named entities

The `ner` component finds names of people (`PERSON`), places (`GPE`, `LOC`), organizations (`ORG`),
nationalities and political or religious groups (`NORP`), dates, amounts and more. `displacy` shows
them in the text:

In [ ]:
example = docs[-2][:120]                     # the first 120 tokens of the 2021 speech
displacy.render(example, style="ent", jupyter=True)

In [ ]:
ents = pd.DataFrame([{"year": year, "text": e.text, "label": e.label_}
                     for year, doc in zip(speeches["year"], docs) for e in doc.ents])
print(ents["label"].value_counts().head(10).to_string())
print()
for label in ["GPE", "NORP", "ORG"]:
    top = ents[ents.label == label]["text"].value_counts().head(8)
    print(f"{label:<5}", ", ".join(f"{t} ({n})" for t, n in top.items()))

Look critically at the lists: NER models make mistakes, and some entities have several spellings
("America", "the United States", "the United States of America"). For real analyses, check a sample
of each entity type and merge variants.

> **✏️ Exercise 3**
>
> Which places do presidents talk about? For each century (1700s/1800s, 1900s, 2000s), list the ten most
> frequent `GPE` entities, excluding variants of the United States/America. What changes?

In [ ]:
# ✅ Solution
US = {"america", "the united states", "united states", "the united states of america", "u.s.", "usa", "nation"}
ents["century"] = pd.cut(ents.year, [1700, 1899, 1999, 2100], labels=["1789-1899", "1900-1999", "2000-2025"])
gpe = ents[(ents.label == "GPE") & (~ents.text.str.lower().isin(US))]
for century, group in gpe.groupby("century", observed=True):
    print(f"{century}:", ", ".join(group.text.value_counts().head(10).index))
# Early speeches name few foreign places; in the 20th century other countries, regions and cities
# appear more often, reflecting a larger role in the world. Expect some noise: words wrongly tagged as
# GPE, and the same place spelled in different ways.

## 7. Dependencies: who does what to whom

The **dependency parser** links every word to its *head*, with a label for the relation. The main
verb is the root; `nsubj` marks the subject, `dobj` the direct object, `prep` and `pobj` a
prepositional phrase, and so on.

In [ ]:
sentence = nlp("The government must protect the rights of every citizen.")
displacy.render(sentence, style="dep", jupyter=True, options={"compact": True, "distance": 110})
for t in sentence:
    print(f"{t.text:<10} {t.dep_:<6} head: {t.head.text}")

From the tree we can extract **subject-verb-object triples**: for every verb, its subject and its
object. That turns sentences into "who does what to whom", which you can aggregate over a corpus.

In [ ]:
def svo_triples(doc):
    triples = []
    for verb in doc:
        if verb.pos_ not in ("VERB", "AUX"):
            continue
        subjects = [c for c in verb.children if c.dep_ in ("nsubj", "nsubjpass")]
        objects = [c for c in verb.children if c.dep_ in ("dobj", "attr")]
        for s in subjects:
            for o in objects:
                triples.append((s.lemma_.lower(), verb.lemma_.lower(), o.lemma_.lower()))
    return triples


print(svo_triples(sentence))
triples = pd.DataFrame([(year, *t) for year, doc in zip(speeches["year"], docs) for t in svo_triples(doc)],
                       columns=["year", "subject", "verb", "object"])
print(len(triples), "triples, e.g.:")
triples.sample(8, random_state=1)

In [ ]:
print('What does "we" do? (most common verbs and objects with "we" as subject)')
print(triples[triples.subject == "we"].groupby(["verb", "object"]).size().nlargest(12).to_string())

> **✏️ Exercise 4**
>
> 1. What does **government** do in these speeches? List the most common verbs with `government` as
>    subject.
> 2. What happens **to** freedom (or liberty)? List the verbs with `freedom` or `liberty` as object.
> 3. Compare the verbs with subject `government` before and after 1933 (Franklin Roosevelt). Do
>    you see a change in how presidents describe government?

In [ ]:
# ✅ Solution
gov = triples[triples.subject == "government"]
print("government does:", gov.verb.value_counts().head(10).to_dict())
free = triples[triples.object.isin(["freedom", "liberty"])]
print("done to freedom/liberty:", free.verb.value_counts().head(10).to_dict())
print("government, before 1933:", gov[gov.year < 1933].verb.value_counts().head(8).to_dict())
print("government, 1933 and later:", gov[gov.year >= 1933].verb.value_counts().head(8).to_dict())
# The counts are small (government is often the subject of passive or complex sentences that our
# simple function skips), so treat this as exploration and read the sentences behind the triples.
# With only direct subject-verb-object links, we miss "freedom must be defended" (a passive with no
# object) and much more: real projects use richer patterns, or a parser-based tool built for it.

### Noun chunks

`doc.noun_chunks` gives base noun phrases ("the new government", "every citizen"): a quick way to
find multi-word concepts.

In [ ]:
chunks = Counter(c.text.lower() for doc in docs for c in doc.noun_chunks
                 if len(c) > 1 and not all(t.is_stop for t in c))
print(chunks.most_common(20))

## 8. Rule-based matching

The **`Matcher`** finds token sequences described by their attributes, which is far more powerful
than a regular expression on raw text. A pattern is a list of dictionaries, one per token. This one
finds promises and obligations: "we" + a modal verb + any verb, with an optional adverb in between
("we will *always* defend").

In [ ]:
matcher = Matcher(nlp.vocab)
matcher.add("WE_MODAL_VERB", [[
    {"LOWER": "we"},
    {"LOWER": {"IN": ["will", "shall", "must", "can"]}},
    {"POS": "ADV", "OP": "?"},                              # optional adverb
    {"POS": "VERB"},
]])

promises = []
for year, doc in zip(speeches["year"], docs):
    for match_id, start, end in matcher(doc):
        span = doc[start:end]
        promises.append({"year": year, "modal": span[1].lower_, "verb": span[-1].lemma_, "match": span.text})
promises = pd.DataFrame(promises)
print(len(promises), "matches")
print(promises.verb.value_counts().head(12).to_string())

A **`PhraseMatcher`** finds exact phrases from a list, quickly, even with thousands of terms (for
example, a dictionary from Day 1):

In [ ]:
phrases = PhraseMatcher(nlp.vocab, attr="LOWER")               # case-insensitive
phrases.add("ECONOMY", [nlp.make_doc(t) for t in ["economy", "jobs", "taxes", "trade", "industry", "prosperity"]])
phrases.add("SECURITY", [nlp.make_doc(t) for t in ["war", "peace", "army", "defense", "security", "enemies"]])

counts = pd.DataFrame([{"year": year, **Counter(nlp.vocab.strings[m] for m, s, e in phrases(doc))}
                       for year, doc in zip(speeches["year"], docs)]).fillna(0).set_index("year")
counts.tail(8)

> **✏️ Exercise 5**
>
> 1. Add a second pattern to a new `Matcher` that also catches "**our** nation/country/people **must/will**
>    + verb" (e.g., "our nation will endure"). Use `{"LOWER": {"IN": [...]}}` for the noun.
> 2. Split the `WE_MODAL_VERB` matches into speeches before and after 1945. Which modal verbs
>    (`will`, `shall`, `must`, `can`) become more or less common, per speech?

In [ ]:
# ✅ Solution
matcher2 = Matcher(nlp.vocab)
matcher2.add("OUR_NOUN_MODAL_VERB", [[
    {"LOWER": "our"},
    {"LOWER": {"IN": ["nation", "country", "people"]}},
    {"LOWER": {"IN": ["will", "shall", "must", "can"]}},
    {"POS": "ADV", "OP": "?"},
    {"POS": "VERB"},
]])
found = [doc[s:e].text for doc in docs for _, s, e in matcher2(doc)]
print(len(found), "matches, e.g.:", found[:6])

promises["period"] = promises.year.map(lambda y: "before 1945" if y < 1945 else "1945 and later")
n_speeches = speeches.year.map(lambda y: "before 1945" if y < 1945 else "1945 and later").value_counts()
table = promises.groupby(["period", "modal"]).size().unstack(fill_value=0)
print("Matches per speech:")
display(table.div(n_speeches, axis=0).round(2))
print("Share of each modal among the matches:")
display(table.div(table.sum(axis=1), axis=0).round(2))
# Per speech, every modal becomes more common after 1945, because "we + modal + verb" as a whole
# becomes far more frequent (more "we", see Exercise 2). The shares show the change in the mix:
# "will" takes over, "shall" and "must" lose ground. Part of that is a general change in English,
# not only in speeches; compare with a reference corpus before interpreting it.

## 9. Dutch, and working at scale

### Dutch

The same code works for Dutch with a Dutch model. Lemmatization matters more for Dutch than for
English, because Dutch inflects more (and has separable verbs like *opbellen*: "ik bel je op").

In [ ]:
nlp_nl = spacy.load("nl_core_news_sm")
doc_nl = nlp_nl("De boeren uit Gelderland reden dinsdag met hun trekkers naar Den Haag, omdat de "
                "minister de nieuwe stikstofregels niet goed had uitgelegd.")
display(pd.DataFrame([{"text": t.text, "lemma": t.lemma_, "pos": t.pos_, "dep": t.dep_} for t in doc_nl]))
print("Entities:", [(e.text, e.label_) for e in doc_nl.ents])

### Working at scale

Processing millions of documents takes planning:

- **`nlp.pipe(texts, batch_size=...)`** instead of a loop over `nlp(text)`.
- **Switch off what you don't need**: `nlp.pipe(texts, disable=["ner", "parser"])` if you only need
  lemmas and tags, or `spacy.load("en_core_web_sm", exclude=["ner"])`. The parser is the slowest part.
- **Several processes**: `nlp.pipe(texts, n_process=4)` on a machine with several cores.
- **Save processed docs** with `DocBin` instead of re-parsing every time.
- **Long texts**: spaCy has a `max_length` (1,000,000 characters by default); split very long documents
  into paragraphs.

In [ ]:
texts = list(speeches["text"])
for label, kwargs in [("full pipeline", {}), ("without parser and NER", {"disable": ["parser", "ner"]})]:
    start = time.time()
    _ = list(nlp.pipe(texts, batch_size=8, **kwargs))
    print(f"{label:<25} {time.time() - start:5.1f} s")

In [ ]:
from spacy.tokens import DocBin

DocBin(store_user_data=True, docs=docs).to_disk("inaugural.spacy")       # save once...
reloaded = list(DocBin().from_disk("inaugural.spacy").get_docs(nlp.vocab))  # ...reload in seconds
print(len(reloaded), "docs reloaded; first entity:", reloaded[0].ents[0], reloaded[0].ents[0].label_)

### Further reading

- Honnibal, M., Montani, I., Van Landeghem, S., & Boyd, A. (2020). *spaCy: Industrial-strength natural
  language processing in Python*. Zenodo. https://doi.org/10.5281/zenodo.1212303
- van Atteveldt, W., Trilling, D., & Arcíla Calderón, C. (2022). *Computational analysis of
  communication*. Wiley. (Free online at cssbook.net.)
- The free, interactive spaCy course: https://course.spacy.io